In [1]:
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")
cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [9]:
!hdfs dfs -mkdir -p /user/nabill/ecommerce/raw
!hdfs dfs -mkdir -p /user/nabill/ecommerce/processed
!hdfs dfs -ls -R /user/nabill/ecommerce

drwxr-xr-x   - nabill supergroup          0 2026-09-09 22:22 /user/nabill/ecommerce/processed
drwxr-xr-x   - nabill supergroup          0 2026-09-09 22:25 /user/nabill/ecommerce/raw
-rw-r--r--   1 nabill supergroup      12329 2026-09-09 22:25 /user/nabill/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 nabill supergroup      12684 2026-09-09 22:25 /user/nabill/ecommerce/raw/transaksi_yogyakarta.csv


In [10]:
!hdfs dfs -put transaksi_magelang.csv /user/nabill/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/nabill/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/nabill/ecommrce/raw/
!hdfs dfs -ls -h /user/nabill/ecommerce/raw

put: `/user/nabill/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/nabill/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/nabill/ecommrce/raw/': No such file or directory: `hdfs://localhost:9000/user/nabill/ecommrce/raw'
Found 2 items
-rw-r--r--   1 nabill supergroup     12.0 K 2026-09-09 22:25 /user/nabill/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 nabill supergroup     12.4 K 2026-09-09 22:25 /user/nabill/ecommerce/raw/transaksi_yogyakarta.csv


In [15]:
import pandas as pd
import io

def read_csv_from_hdfs(hdfs_path):
    cmd_output = !hdfs dfs -cat {hdfs_path}
    csv_data = "\n".join(cmd_output)
    return pd.read_csv(io.StringIO(csv_data))

df_magelang = read_csv_from_hdfs("/user/nabill/ecommerce/raw/transaksi_magelang.csv")
df_yogyakarta = read_csv_from_hdfs("/user/nabill/ecommerce/raw/transaksi_yogyakarta.csv")
df_semarang = read_csv_from_hdfs("/user/nabill/ecommerce/raw/transaksi_semarang.csv")

df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)

print("=== Hasil Verifikasi Gabungan Data Kota ===")
print(df_gabungan["kota"].value_counts())

=== Hasil Verifikasi Gabungan Data Kota ===
kota
Magelang      200
Yogyakarta    200
Name: count, dtype: int64


In [18]:
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_ringkasan = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()

df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

!hdfs dfs -put -f data_gabungan_bersih.csv /user/nabill/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/nabill/ecommerce/processed/
!hdfs dfs -ls -h /user/nabill/ecommerce/processed/

Found 2 items
-rw-r--r--   1 nabill supergroup     29.9 K 2026-09-10 04:36 /user/nabill/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 nabill supergroup        387 2026-09-10 04:36 /user/nabill/ecommerce/processed/ringkasan_kota_kategori.csv


Menyimpan data mentah atau raw secara terpisah dari data olahan (processed) di HDFS adalah praktik yang baik di Biga Data. Keuntungannya adalah menjaga imutabilitas data mentah sebagai "Single Source of Truth". Misalnya di kemudian hari ada kesalahan logika di algoritma pemrosesan data atau ketika tim analisis membutuhkan metrik analisis baru yang datanya tidak ada di data olahan saat ini, mereka bisa memproses ulsng dsts mentah yang asli itu dari awal tanpa takut datanya hilang. Selain itu, pemisahan ini juga mempermudah manajemen hak akses keamanan dan efisiensi komputer, jadi bisa fokus mengelola data di area raw bagi data engineer, sedangkan data anlyst cukup hanya mengakses area processed agar query cepat, ini juga bermanfaat supaya data tidak tercampur berantakan, risiko terhapusnya data secara tidak sengaja juga sangat kecil, dan juga menjaga struktur tata kelola data agar tetap rapi walaupun nanti datanya terus berkebang.